In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (classification_report, roc_auc_score,
                             average_precision_score, confusion_matrix,
                             precision_recall_curve, RocCurveDisplay,
                             PrecisionRecallDisplay, f1_score)


In [ ]:
df = pd.read_csv('../data/kenya_synthetic_loans.csv')
print(df.shape)
print(df.columns.tolist())
display(pd.concat([df.head(), df.tail()]))

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()

n = len(numeric_cols)
ncols = 3
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.5 * nrows))
axes = axes.flat

for ax, col in zip(axes, numeric_cols):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(col, fontsize=9)

for ax in axes[n:]:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
df = df.copy()
df["mpesa_to_income"] = df["mpesa_avg_monthly_inflow_kes"] / df["monthly_income_kes"].clip(lower=1)
df["airtime_to_income"] = df["airtime_spend_monthly_kes"] / df["monthly_income_kes"].clip(lower=1)
df["loan_per_mpesa"] = df["loan_amount_kes"] / df["mpesa_avg_monthly_inflow_kes"].clip(lower=1)
df["has_prior_default"] = (df["prior_defaults"] > 0).astype(int)
df["total_prior_loans"] = df["prior_loans"] + df["active_loans_elsewhere"]

y = df["defaulted_90dpd"]
groups = df["borrower_id"]
X = df.drop(columns=["defaulted_90dpd", "loan_id", "borrower_id"])

cat_cols = ["county", "employment_type", "channel"]
num_cols = [c for c in X.columns if c not in cat_cols]
print(f"numeric: {len(num_cols)}  categorical: {len(cat_cols)}")


In [ ]:
pre = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=10), cat_cols),
])


In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
tr_idx, te_idx = next(gss.split(X, y, groups=groups))

X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]
assert not set(groups.iloc[tr_idx]) & set(groups.iloc[te_idx])
print(f"train: {len(X_tr)}  test: {len(X_te)}  default rate tr/test: {y_tr.mean():.3f}/{y_te.mean():.3f}")


In [ ]:
rf = RandomForestClassifier(
    n_estimators=500,
    min_samples_leaf=3,
    max_features="sqrt",
    class_weight="balanced_subsample",
    oob_score=True,
    n_jobs=-1,
    random_state=42,
)

pipe = Pipeline([("pre", pre), ("rf", rf)])
pipe.fit(X_tr, y_tr)
print(f"OOB score: {pipe.named_steps['rf'].oob_score_:.3f}")


In [ ]:
y_prob = pipe.predict_proba(X_te)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

print(classification_report(y_te, y_pred, digits=3))
print(f"ROC-AUC: {roc_auc_score(y_te, y_prob):.3f}")
print(f"PR-AUC : {average_precision_score(y_te, y_prob):.3f} (baseline = {y_te.mean():.3f})")
print(confusion_matrix(y_te, y_pred))


In [ ]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
cv = cross_validate(
    pipe,
    X,
    y,
    groups=groups,
    cv=sgkf,
    scoring={"roc_auc": "roc_auc", "pr_auc": "average_precision"},
    n_jobs=-1,
)
print("CV ROC-AUC: %.3f ± %.3f" % (cv["test_roc_auc"].mean(), cv["test_roc_auc"].std()))
print("CV PR-AUC : %.3f ± %.3f" % (cv["test_pr_auc"].mean(), cv["test_pr_auc"].std()))


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
RocCurveDisplay.from_predictions(y_te, y_prob, ax=ax[0])
PrecisionRecallDisplay.from_predictions(y_te, y_prob, ax=ax[1])
ax[1].axhline(y_te.mean(), ls="--", c="grey", label="baseline")
ax[1].legend()
plt.tight_layout()
plt.show()

ohe = pipe.named_steps["pre"].named_transformers_["cat"]
feature_names = num_cols + list(ohe.get_feature_names_out(cat_cols))
importances = pd.Series(
    pipe.named_steps["rf"].feature_importances_, index=feature_names
).sort_values(ascending=False).head(15)
print(importances)
importances.sort_values().plot.barh(figsize=(7, 5))
plt.title("Top 15 features")
plt.show()


In [ ]:
thresholds = np.linspace(0.05, 0.7, 66)
f1_scores = [f1_score(y_te, (y_prob >= threshold).astype(int)) for threshold in thresholds]
best_f1_threshold = thresholds[int(np.argmax(f1_scores))]
print(f"Best threshold by F1: {best_f1_threshold:.2f}")

amounts = X_te["loan_amount_kes"].to_numpy()
costs = []
for threshold in thresholds:
    predictions = (y_prob >= threshold).astype(int)
    false_negatives = ((y_te == 1) & (predictions == 0)).to_numpy()
    false_positives = ((y_te == 0) & (predictions == 1)).to_numpy()
    costs.append(amounts[false_negatives].sum() + 0.1 * amounts[false_positives].sum())
best_cost_threshold = thresholds[int(np.argmin(costs))]
print(f"Best threshold by cost: {best_cost_threshold:.2f}")
